# 🎬 Movie Recommender System — Full 45K Dataset Version
### Dataset: https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset

**Files needed in same folder as this notebook:**
- `movies_metadata.csv`
- `keywords.csv`
- `credits.csv`

## 1️⃣ Install & Import Libraries

In [ ]:
!pip install nltk scikit-learn pandas

In [ ]:
import pandas as pd
import ast
import nltk
import pickle
import warnings
warnings.filterwarnings('ignore')

from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt')
ps = PorterStemmer()

print('✅ Libraries loaded!')

## 2️⃣ Load the 3 CSV Files

In [ ]:
# Load all 3 files
meta     = pd.read_csv('movies_metadata.csv', low_memory=False)
keywords = pd.read_csv('keywords.csv')
credits  = pd.read_csv('credits.csv')

print(f'Metadata shape : {meta.shape}')
print(f'Keywords shape : {keywords.shape}')
print(f'Credits shape  : {credits.shape}')
meta.head(2)

## 3️⃣ Clean & Merge

In [ ]:
# The id column in metadata has some bad rows (e.g. '-', 'tt...') — drop them
meta = meta[meta['id'].apply(lambda x: str(x).isdigit())].copy()
meta['id'] = meta['id'].astype(int)

keywords['id'] = keywords['id'].astype(int)
credits['id']  = credits['id'].astype(int)

# Merge all three on id
movies = meta.merge(keywords, on='id').merge(credits, on='id')

print(f'Merged shape: {movies.shape}')
movies[['title', 'genres', 'keywords', 'cast', 'crew', 'overview']].head(2)

## 4️⃣ Select Important Columns & Drop Nulls

In [ ]:
movies = movies[['id', 'title', 'genres', 'keywords', 'cast', 'crew', 'overview']].copy()

print('Null values before:')
print(movies.isnull().sum())

movies.dropna(inplace=True)
movies.drop_duplicates(subset='title', inplace=True)

print(f'\n✅ Total clean movies: {len(movies)}')

## 5️⃣ Feature Engineering
> This dataset stores genres/keywords/cast/crew as JSON strings — we parse them the same way as before.

In [ ]:
def safe_parse(text):
    """Safely parse a JSON string — returns empty list on failure."""
    try:
        return ast.literal_eval(str(text))
    except:
        return []

def get_names(text):
    """Extract all names from parsed JSON list."""
    parsed = safe_parse(text)
    return [i['name'].replace(' ', '') for i in parsed if 'name' in i]

def get_cast(text):
    """Extract top-3 cast names."""
    parsed = safe_parse(text)
    return [i['name'].replace(' ', '') for i in parsed[:3] if 'name' in i]

def get_director(text):
    """Extract director name from crew."""
    parsed = safe_parse(text)
    return [i['name'].replace(' ', '') for i in parsed if i.get('job') == 'Director']

def split_overview(text):
    """Split overview string into word list."""
    return str(text).split()

# Apply all
movies['genres']   = movies['genres'].apply(get_names)
movies['keywords'] = movies['keywords'].apply(get_names)
movies['cast']     = movies['cast'].apply(get_cast)
movies['crew']     = movies['crew'].apply(get_director)
movies['overview'] = movies['overview'].apply(split_overview)

print('✅ Feature engineering done!')
movies[['title','genres','keywords','cast','crew']].head(3)

## 6️⃣ Build Tags WITH Genre & Keyword Boosting
> Genres and keywords are repeated **3×** so emotional tone (Romance, Drama) has more weight than actor names.

In [ ]:
movies['tags'] = (
    movies['overview']
    + movies['genres']   * 3    # genres weighted 3×
    + movies['keywords'] * 3    # keywords weighted 3×
    + movies['cast']            # cast once
    + movies['crew']            # director once
)

new_df = movies[['id', 'title', 'tags']].copy()
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
new_df.reset_index(drop=True, inplace=True)

print(f'✅ Tags built! Total movies: {len(new_df)}')

## 7️⃣ Stemming
> Converts `romantic`, `romance`, `romancing` → all become `romanc` → much better similarity scores

In [ ]:
def stem(text):
    return " ".join([ps.stem(word) for word in text.split()])

print('Applying stemming to all movies... (this may take ~30 seconds)')
new_df['tags'] = new_df['tags'].apply(stem)
print('✅ Stemming done!')

## 8️⃣ Vectorize & Compute Similarity

In [ ]:
print('Building TF-IDF vectors... (may take ~1 minute for 45k movies)')
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
vectors = tfidf.fit_transform(new_df['tags']).toarray()
print(f'✅ Vectors shape: {vectors.shape}')

print('Computing cosine similarity... (may take 2-3 minutes)')
similarity = cosine_similarity(vectors)
print(f'✅ Similarity matrix shape: {similarity.shape}')

## 9️⃣ Recommend Function

In [ ]:
def recommend(movie, n=10):
    """
    Recommend n movies similar to the given title.
    Partial/case-insensitive match supported.
    """
    # Exact match first
    match = new_df[new_df['title'].str.lower() == movie.lower()]

    # Partial match fallback
    if len(match) == 0:
        match = new_df[new_df['title'].str.contains(movie, case=False, na=False)]

    if len(match) == 0:
        print(f'❌ Movie "{movie}" not found in dataset.')
        return

    movie_title = match.iloc[0]['title']
    movie_index = match.index[0]

    distances  = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:n+1]

    print(f'🎬 Because you liked: "{movie_title}"')
    print(f'📋 Top {n} Recommendations:')
    print('-' * 45)
    for rank, (idx, score) in enumerate(movies_list, 1):
        print(f'  {rank:2}. {new_df.iloc[idx].title}  (score: {score:.3f})')

print('✅ Ready!')

## 🔟 Check Which Romantic Movies Are Now in the Dataset

In [ ]:
films_to_check = [
    'The Fault in Our Stars',
    'Me Before You',
    'A Walk to Remember',
    'The Notebook',
    'Five Feet Apart',
    'Everything Everything',
    'Love Rosie',
    'P.S. I Love You',
    'Dear John',
    'Safe Haven',
    'If I Stay',
    'The Lucky One'
]

print('📋 Dataset Coverage Check:')
print('-' * 50)
for film in films_to_check:
    found = new_df[new_df['title'].str.lower() == film.lower()]
    status = '✅ FOUND' if len(found) > 0 else '❌ NOT IN DATASET'
    print(f'  {status}  →  {film}')

## 1️⃣1️⃣ Test Recommendations

In [ ]:
recommend('The Fault in Our Stars')

In [ ]:
recommend('Me Before You')

In [ ]:
recommend('The Notebook')

In [ ]:
recommend('Avatar')

In [ ]:
recommend('Batman Begins')

## 1️⃣2️⃣ Save Model Files

In [ ]:
pickle.dump(new_df, open('movies.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
print('✅ movies.pkl and similarity.pkl saved!')